# 📡 NSE Trade Planner: Nifty 500 + live checks
* **Evening (after 3:35 PM):** `Runtime → Run all` → exact plan for tomorrow: what to BUY (price, qty, stop-loss), the watchlist with **exact BUY trigger prices**, and **exact sell levels for the stocks you own**.
* **During market hours:** re-run only section 7 (*Live check*) as often as you like.

> ⚠️ Educational tool, not financial advice. Paper trade for 2–3 months first.

## 1. Install

In [ ]:
!pip install yfinance -q

## 2. Your settings

In [ ]:
# ============================ YOUR SETTINGS ============================
UNIVERSE       = "NIFTY500"    # "NIFTY50", "NIFTY100", "NIFTY200" or "NIFTY500"
HISTORY_START  = "2015-01-01"  # long history = fairer statistics

CAPITAL        = 100_000       # your TOTAL trading money (₹)
MAX_POSITIONS  = 5             # max stocks held at once -> each slot = CAPITAL / MAX_POSITIONS
RISK_PER_TRADE = 0.01          # max loss per trade if stop-loss hits (1% of CAPITAL)

MIN_TURNOVER_CR = 10           # skip stocks trading < ₹10 crore/day on average (hard to buy/sell)
NEAR_BUY_PCT    = 3.0          # watchlist: BUY trigger price within 3% of today's close
HOLDINGS_FILE   = "my_holdings.csv"   # the stocks YOU actually bought (Symbol,BuyDate,BuyPrice,Qty)

# "Strategy works on this stock?" filter (from its own past backtest)
MIN_SHARPE     = 0.5
MIN_TRADES     = 10

CONFIG = {
    "ema_fast": 20,            # short-term trend line
    "ema_slow": 50,            # medium-term trend line
    "rsi_period": 14,
    "rsi_max_entry": 70,       # skip buys when RSI > 70 (overbought)
    "atr_period": 14,
    "atr_stop_mult": 2.0,      # stop-loss distance = 2 x ATR below price
    "max_gap_atr": 0.5,        # don't chase: skip if next open > close + 0.5 x ATR
    "cost_pct": 0.0015,        # ~0.15% per side (STT 0.1% + stamp + exchange + GST + DP)
}

## 3. Strategy engine (no need to edit)

In [ ]:
import io, os, json, time, datetime as dt
import numpy as np
import pandas as pd

SLOT        = CAPITAL / MAX_POSITIONS         # money per stock
RISK_BUDGET = CAPITAL * RISK_PER_TRADE        # max ₹ loss per trade


# ------------------------------ indicators ------------------------------
def add_indicators(df, c=CONFIG):
    df = df.copy()
    df["ema_fast"] = df["Close"].ewm(span=c["ema_fast"], adjust=False).mean()
    df["ema_slow"] = df["Close"].ewm(span=c["ema_slow"], adjust=False).mean()
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).ewm(alpha=1 / c["rsi_period"], adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1 / c["rsi_period"], adjust=False).mean()
    df["rsi"] = 100 - 100 / (1 + gain / loss.replace(0, np.nan))
    df["ag"], df["al"] = gain, loss              # kept to compute trigger prices
    pc = df["Close"].shift()
    tr = pd.concat([df["High"] - df["Low"], (df["High"] - pc).abs(), (df["Low"] - pc).abs()], axis=1).max(axis=1)
    df["atr"] = tr.ewm(alpha=1 / c["atr_period"], adjust=False).mean()
    df = df.dropna()
    up = (df["ema_fast"] > df["ema_slow"]) & (df["ema_fast"].shift() <= df["ema_slow"].shift())
    dn = (df["ema_fast"] < df["ema_slow"]) & (df["ema_fast"].shift() >= df["ema_slow"].shift())
    df["buy"] = up & (df["rsi"] < c["rsi_max_entry"])
    df["sell"] = dn
    return df


# ------------------------------ backtest ------------------------------
def backtest(df, c=CONFIG, slot=None, risk_budget=None):
    """
    Exactly the rules the daily plan tells you to follow:
      * signal decided on day t's CLOSE (after 3:30 PM)
      * BUY next morning at the open with a LIMIT = close + 0.5*ATR (skip if it opens higher)
      * stop-loss = entry - 2*ATR, raised every evening to (close - 2*ATR), never lowered
      * exit at stop (or at the open if it gaps below), or at next open after EMA cross-down
    Returns equity curve, trade list and the open position (if any) for tomorrow's plan.
    """
    slot = slot or SLOT
    risk_budget = risk_budget or RISK_BUDGET
    m, cost = c["atr_stop_mult"], c["cost_pct"]
    O, H, L, C = (df[k].values for k in ("Open", "High", "Low", "Close"))
    A, B, S, D = df["atr"].values, df["buy"].values, df["sell"].values, df.index
    cash, qty, entry_px, stop, entry_i = slot, 0, 0.0, 0.0, None
    trades, equity, skipped = [], [], 0

    def close_pos(j, px, why):
        nonlocal cash, qty
        cash += qty * px * (1 - cost)
        pnl = qty * (px * (1 - cost) - entry_px * (1 + cost))
        trades.append({"entry_date": D[entry_i], "entry": entry_px, "exit_date": D[j], "exit": px,
                       "qty": qty, "reason": why, "pnl": pnl,
                       "ret_%": (px * (1 - cost) / (entry_px * (1 + cost)) - 1) * 100,
                       "days": j - entry_i})
        qty = 0

    for i in range(len(df) - 1):
        j = i + 1
        if qty > 0:                                   # 1) stop-loss check
            stop = max(stop, C[i] - m * A[i])
            if L[j] <= stop:
                close_pos(j, min(O[j], stop), "STOP")
        if qty > 0 and S[i]:                          # 2) trend reversal exit
            close_pos(j, O[j], "EMA cross down")
        elif qty == 0 and B[i]:                       # 3) entry
            limit = C[i] + c["max_gap_atr"] * A[i]
            if O[j] > limit:
                skipped += 1
            else:
                risk_ps = m * A[i]
                size = min(int(risk_budget / risk_ps), int(cash / (O[j] * (1 + cost))))
                if size > 0:
                    qty, entry_px, entry_i = size, O[j], j
                    stop = entry_px - risk_ps
                    cash -= qty * entry_px * (1 + cost)
                    if L[j] <= stop:                  # stopped out the same day
                        close_pos(j, stop, "STOP")
        equity.append((D[j], cash + qty * C[j]))

    position = None
    if qty > 0:
        n = len(df) - 1
        position = {"entry_date": D[entry_i], "entry": entry_px, "qty": qty,
                    "stop_tomorrow": max(stop, C[n] - m * A[n]), "days": n - entry_i + 1}
    eq = pd.Series(dict(equity))
    return eq, pd.DataFrame(trades), position, skipped


def stats(eq, tr, df, slot=None):
    slot = slot or SLOT
    years = max((eq.index[-1] - eq.index[0]).days / 365.25, 0.1)
    r = eq.pct_change().dropna()
    bh = df["Close"].iloc[-1] / df["Close"].loc[eq.index[0]:].iloc[0]
    n = len(tr)
    in_mkt = tr["days"].sum() / max(len(eq), 1) * 100 if n else 0.0
    return {
        "Strategy CAGR %": ((eq.iloc[-1] / slot) ** (1 / years) - 1) * 100,
        "Buy&Hold CAGR %": (bh ** (1 / years) - 1) * 100,
        "Sharpe": np.sqrt(252) * r.mean() / r.std() if r.std() > 0 else 0.0,
        "MaxDD %": (eq / eq.cummax() - 1).min() * 100,
        "Trades": n,
        "Win %": (tr["pnl"] > 0).mean() * 100 if n else 0.0,
        "Avg win %": tr.loc[tr["pnl"] > 0, "ret_%"].mean() if n and (tr["pnl"] > 0).any() else 0.0,
        "Avg loss %": tr.loc[tr["pnl"] <= 0, "ret_%"].mean() if n and (tr["pnl"] <= 0).any() else 0.0,
        "Avg hold days": tr["days"].mean() if n else 0.0,
        "Time in market %": in_mkt,
    }


# ------------------------------ tomorrow's plan ------------------------------


def next_trading_day(d):
    return (pd.Timestamp(d) + pd.offsets.BDay(1)).date()


# ------------------------------ trigger prices ------------------------------
def cross_price(row, c=CONFIG):
    """Closing price at which EMA fast and EMA slow become equal on the NEXT day.
    No position: close ABOVE it -> BUY signal.  In a position: close BELOW it -> SELL signal."""
    af, as_ = 2 / (c["ema_fast"] + 1), 2 / (c["ema_slow"] + 1)
    return ((1 - as_) * row["ema_slow"] - (1 - af) * row["ema_fast"]) / (af - as_)


def rsi_cap_price(row, c=CONFIG):
    """Highest next-day close that keeps RSI below rsi_max_entry (above it, a BUY is blocked)."""
    a, k = 1 / c["rsi_period"], c["rsi_max_entry"] / (100 - c["rsi_max_entry"])
    return row["Close"] + (k * (1 - a) * row["al"] - (1 - a) * row["ag"]) / a


def turnover_cr(df, n=20):
    return float((df["Close"] * df["Volume"]).tail(n).mean() / 1e7)


# ------------------------------ strategy plan (every stock) ------------------------------
def trade_plan(sym, df, tr, position, c=CONFIG):
    last, day = df.iloc[-1], df.index[-1]
    nxt = next_trading_day(day).strftime("%a %d-%b")
    m, close, atr = c["atr_stop_mult"], last["Close"], last["atr"]
    trig, cap = cross_price(last, c), rsi_cap_price(last, c)
    p = {"Symbol": sym, "Close": round(close, 2), "RSI": round(last["rsi"], 1), "Next day": nxt,
         "Buy limit": None, "Qty": None, "Stop-loss": None, "Risk ₹": None, "BUY trigger": None,
         "RSI cap": round(cap, 2), "Exit trigger": None, "Trigger dist %": None,
         "Entry": None, "Entry date": None, "Day #": None, "P/L %": None, "Plan": ""}
    exited_today = len(tr) and tr.iloc[-1]["exit_date"] == day

    if position:
        e = position
        pl = (close / e["entry"] - 1) * 100
        p.update({"Entry": round(e["entry"], 2), "Entry date": e["entry_date"].date(), "Day #": e["days"],
                  "P/L %": round(pl, 2), "Stop-loss": round(e["stop_tomorrow"], 2), "Qty": e["qty"]})
        if last["sell"]:
            p["Action"] = "🔴 SELL"
            p["Plan"] = (f"Trend reversed today. SELL all on {nxt} at the open (9:15 AM) and delete the GTT. "
                         f"Result: {pl:+.1f}% in {e['days']} days.")
        else:
            p["Action"] = "🟡 HOLD"
            p["Exit trigger"] = round(trig, 2)
            p["Trigger dist %"] = round((trig / close - 1) * 100, 2)
            p["Plan"] = (f"Day {e['days']} (strategy bought {e['entry_date'].date()} @ ₹{e['entry']:.2f}, {pl:+.1f}%). "
                         f"SELL if (1) price touches ₹{e['stop_tomorrow']:.2f} any time (GTT stop-loss), or "
                         f"(2) it CLOSES below ₹{trig:.2f} (trend exit → sell next open). "
                         f"Didn't buy on the signal day? Don't buy now.")
    elif exited_today:
        t = tr.iloc[-1]
        p["Action"] = "🛑 STOPPED" if t["reason"] == "STOP" else "⚪ EXITED"
        p["Plan"] = (f"Strategy exited today at ₹{t['exit']:.2f} ({t['reason']}, {t['ret_%']:+.1f}%). "
                     f"Still holding? SELL at the open on {nxt}.")
    elif last["buy"]:
        risk_ps, limit = m * atr, close + c["max_gap_atr"] * atr
        qty = min(int(RISK_BUDGET / risk_ps), int(SLOT / limit))
        p.update({"Action": "🟢 BUY", "Buy limit": round(limit, 2), "Qty": qty,
                  "Stop-loss": round(close - risk_ps, 2), "Risk ₹": round(qty * risk_ps)})
        p["Plan"] = (f"On {nxt}, 9:15-9:30 AM: LIMIT BUY (CNC) {qty} shares at up to ₹{limit:.2f}. "
                     f"Opens higher → SKIP. After it fills: GTT stop-loss = your buy price − ₹{risk_ps:.2f} "
                     f"(≈ ₹{close - risk_ps:.2f}). Max loss ≈ ₹{qty * risk_ps:,.0f}."
                     if qty else f"BUY signal, but 1 share (₹{close:,.0f}) exceeds your slot (₹{SLOT:,.0f}). Skip.")
    elif last["ema_fast"] < last["ema_slow"]:
        dist = (trig / close - 1) * 100
        p.update({"BUY trigger": round(trig, 2), "Trigger dist %": round(dist, 2)})
        blocked = trig >= cap
        if 0 < dist <= NEAR_BUY_PCT and not blocked:
            p["Action"] = "👀 NEAR BUY"
            p["Plan"] = (f"A BUY signal fires if it CLOSES between ₹{trig:.2f} and ₹{cap:.2f} on {nxt} "
                         f"(+{dist:.1f}% from today). Set a price alert at ₹{trig:.2f}. Don't buy before the signal.")
        else:
            p["Action"] = "⚪ WAIT"
            p["Plan"] = (f"Downtrend. BUY needs a close above ₹{trig:.2f} ({dist:+.1f}%)"
                         + (" but RSI would be > 70 there, so no signal tomorrow." if blocked else "."))
    else:
        p["Action"] = "⚪ WAIT"
        p["Plan"] = "Uptrend already running, no fresh crossover. Entering late = worse risk/reward. Wait."
    return p


# ------------------------------ YOUR real holdings ------------------------------
def load_holdings(path=None):
    path = path or HOLDINGS_FILE
    if not os.path.exists(path):
        return pd.DataFrame(columns=["Symbol", "BuyDate", "BuyPrice", "Qty"])
    h = pd.read_csv(path, comment="#").dropna(how="all")
    h["Symbol"] = h["Symbol"].str.strip().str.upper().str.replace(".NS", "", regex=False)
    h["BuyDate"] = pd.to_datetime(h["BuyDate"])
    return h


def holding_plan(sym, df, buy_date, buy_price, qty, c=CONFIG):
    """Exact exit plan for a stock YOU own, using your real buy date and price."""
    m = c["atr_stop_mult"]
    after = np.where(df.index >= pd.Timestamp(buy_date))[0]
    if not len(after):
        return {"Symbol": sym, "Action": "❓ CHECK", "Plan": "Buy date is after the latest data. Re-check tomorrow."}
    ei, n = int(after[0]), len(df) - 1
    C, L, A, S = df["Close"].values, df["Low"].values, df["atr"].values, df["sell"].values
    stop = buy_price - m * A[max(ei - 1, 0)]
    hit = sell_day = None
    for j in range(ei, n + 1):
        if j > ei:
            stop = max(stop, C[j - 1] - m * A[j - 1])
        if L[j] <= stop:
            hit = j; break
        if S[j] and sell_day is None:
            sell_day = j
            if j < n:
                break
    last, close = df.iloc[-1], C[n]
    nxt = next_trading_day(df.index[-1]).strftime("%a %d-%b")
    pl = (close / buy_price - 1) * 100
    risk0 = m * A[max(ei - 1, 0)]
    base = {"Symbol": sym, "Qty": int(qty), "Buy price": round(buy_price, 2), "Close": round(close, 2),
            "P/L %": round(pl, 2), "P/L ₹": round((close - buy_price) * qty), "R": round((close - buy_price) / risk0, 2),
            "Day #": n - ei + 1, "Stop-loss": None, "Exit trigger": None}
    if hit is not None:
        d = df.index[hit].date()
        return {**base, "Action": "🛑 STOP HIT",
                "Plan": f"Price touched your stop ₹{stop:.2f} on {d}. Your GTT should have sold. "
                        f"If you still hold it, SELL at the open on {nxt}."}
    if sell_day is not None:
        d = df.index[sell_day].date()
        when = f"at the open on {nxt}" if sell_day == n else "NOW (overdue)"
        return {**base, "Action": "🔴 SELL",
                "Plan": f"Trend exit signal on {d} (EMA{c['ema_fast']} closed below EMA{c['ema_slow']}). "
                        f"SELL all {int(qty)} shares {when} and delete the GTT. Result ≈ {pl:+.1f}%."}
    stop_tm = max(stop, close - m * A[n])
    trig = cross_price(last, c)
    if last["ema_fast"] < last["ema_slow"]:
        base["Stop-loss"] = round(stop_tm, 2)
        return {**base, "Action": "🔴 SELL",
                "Plan": f"Trend is already DOWN (EMA{c['ema_fast']} below EMA{c['ema_slow']}); the system never holds "
                        f"in a downtrend. SELL at the open on {nxt}. (If you insist on holding, keep the GTT stop at ₹{stop_tm:.2f}.)"}
    base.update({"Stop-loss": round(stop_tm, 2), "Exit trigger": round(trig, 2)})
    safe = " Stop is above your buy price: this trade can no longer lose." if stop_tm > buy_price else ""
    return {**base, "Action": "🟡 HOLD",
            "Plan": f"Day {n - ei + 1}, {pl:+.1f}% ({base['R']:+.1f}R). Set GTT stop-loss to ₹{stop_tm:.2f}.{safe} "
                    f"SELL if (1) price touches ₹{stop_tm:.2f} any time (GTT does it), or "
                    f"(2) it CLOSES below ₹{trig:.2f} → sell at next open."}


def run_holdings(prices, path=None):
    h = load_holdings(path)
    rows = []
    for _, r in h.iterrows():
        if r["Symbol"] not in prices:
            rows.append({"Symbol": r["Symbol"], "Action": "❓ CHECK", "Plan": "No price data (check the symbol)."})
            continue
        rows.append(holding_plan(r["Symbol"], add_indicators(prices[r["Symbol"]]), r["BuyDate"], float(r["BuyPrice"]), r["Qty"]))
    return pd.DataFrame(rows).set_index("Symbol") if rows else pd.DataFrame()


# ------------------------------ universe & data ------------------------------
FALLBACK = """ADANIENT ADANIPORTS APOLLOHOSP ASIANPAINT AXISBANK BAJAJ-AUTO BAJFINANCE BAJAJFINSV BEL BHARTIARTL
CIPLA COALINDIA DRREDDY EICHERMOT ETERNAL GRASIM HCLTECH HDFCBANK HDFCLIFE HINDALCO HINDUNILVR ICICIBANK
INDIGO INFY ITC JIOFIN JSWSTEEL KOTAKBANK LT M&M MARUTI MAXHEALTH NESTLEIND NTPC ONGC POWERGRID RELIANCE
SBILIFE SBIN SHRIRAMFIN SUNPHARMA TATACONSUM TATASTEEL TCS TECHM TITAN TRENT ULTRACEMCO WIPRO
DMART HAL DIVISLAB PIDILITIND SIEMENS HAVELLS DABUR GODREJCP BRITANNIA LTIM VEDL TVSMOTOR BANKBARODA PNB
CANBK IOC BPCL GAIL TATAPOWER ADANIGREEN ADANIPOWER AMBUJACEM SHREECEM DLF LODHA ICICIGI ICICIPRULI
CHOLAFIN BAJAJHLDNG PFC RECLTD IRFC ZYDUSLIFE TORNTPHARM MOTHERSON BOSCHLTD NAUKRI INDHOTEL UNITDSPR
VBL JSWENERGY ABB CGPOWER LICI HINDZINC BSE""".split()


def load_universe(name=UNIVERSE, cache_dir="scanner_output"):
    """Official current index list from NSE; falls back to the last saved copy, then a built-in list."""
    import requests
    f = f"ind_{name.lower()}list.csv"
    cache = os.path.join(cache_dir, f)
    for url in (f"https://niftyindices.com/IndexConstituent/{f}",
                f"https://archives.nseindia.com/content/indices/{f}"):
        try:
            r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
            r.raise_for_status()
            u = pd.read_csv(io.StringIO(r.text)).rename(columns={"Company Name": "Company"})
            u = u[["Symbol", "Company", "Industry"]]
            os.makedirs(cache_dir, exist_ok=True); u.to_csv(cache, index=False)
            print(f"✅ Loaded {len(u)} stocks in {name} from NSE")
            return u
        except Exception:
            continue
    if os.path.exists(cache):
        u = pd.read_csv(cache)
        print(f"⚠️ NSE unreachable, using saved {name} list ({len(u)} stocks)")
        return u
    print(f"⚠️ NSE unreachable and no saved list: using built-in {len(FALLBACK)} large caps")
    return pd.DataFrame({"Symbol": FALLBACK, "Company": FALLBACK, "Industry": ""})


def completed_candles(d):
    """Drop today's candle if the market hasn't closed yet (avoids fake signals)."""
    now = pd.Timestamp.now(tz="Asia/Kolkata")
    if len(d) and d.index[-1].date() == now.date() and now.time() < dt.time(15, 35):
        return d.iloc[:-1]
    return d


def download_all(symbols, start=HISTORY_START, batch=50, retries=2):
    import yfinance as yf
    data, todo = {}, [s + ".NS" for s in dict.fromkeys(symbols)]
    for attempt in range(retries + 1):
        failed = []
        for k in range(0, len(todo), batch):
            chunk = todo[k:k + batch]
            try:
                raw = yf.download(chunk, start=start, group_by="ticker", auto_adjust=True,
                                  threads=True, progress=False)
            except Exception:
                failed += chunk; continue
            for t in chunk:
                try:
                    d = completed_candles(raw[t][["Open", "High", "Low", "Close", "Volume"]].dropna())
                    if len(d) > 200:
                        data[t[:-3]] = d
                    elif len(d) == 0:
                        failed.append(t)
                except Exception:
                    failed.append(t)
            print(f"Downloaded {min(k + batch, len(todo))}/{len(todo)} (pass {attempt + 1})")
            time.sleep(1)
        todo = failed
        if not todo:
            break
        time.sleep(5)
    if todo:
        print("⚠️ No data for:", ", ".join(t[:-3] for t in todo))
    return data


def fundamentals(sym):
    import yfinance as yf
    try:
        i = yf.Ticker(sym + ".NS").info
        pm, roe = i.get("profitMargins"), i.get("returnOnEquity")
        return {"Profit margin %": round(pm * 100, 1) if pm is not None else None,
                "ROE %": round(roe * 100, 1) if roe is not None else None,
                "Profitable?": "✅" if (pm or 0) > 0 and (roe or 0) > 0.12 else "❌"}
    except Exception:
        return {"Profit margin %": None, "ROE %": None, "Profitable?": "?"}


# ------------------------------ full scan ------------------------------
ORDER = {"🟢 BUY": 0, "🔴 SELL": 1, "🛑 STOPPED": 2, "🟡 HOLD": 3, "👀 NEAR BUY": 4, "⚪ EXITED": 5, "⚪ WAIT": 6}


def run_scan(universe, prices):
    rows, latest = [], max(d.index[-1] for d in prices.values())
    for sym, raw in prices.items():
        try:
            df = add_indicators(raw)
            eq, tr, pos, _ = backtest(df)
            st = stats(eq, tr, df)
            plan = trade_plan(sym, df, tr, pos)
            to = turnover_cr(df)
            if to < MIN_TURNOVER_CR and plan["Action"] in ("🟢 BUY", "👀 NEAR BUY"):
                plan["Plan"] = f"Signal ignored: only ₹{to:.1f} cr traded per day (min ₹{MIN_TURNOVER_CR} cr). Hard to buy/sell fairly."
                plan["Action"] = "⚪ WAIT"
            works = st["Sharpe"] >= MIN_SHARPE and st["Trades"] >= MIN_TRADES and st["Strategy CAGR %"] > 0
            rows.append({**plan, "Turnover ₹cr": round(to, 1), "Strategy works?": "✅" if works else "❌",
                         **{k: round(v, 2) for k, v in st.items()},
                         "Data date": df.index[-1].date(), "Stale": df.index[-1] < latest})
        except Exception as e:
            print(sym, "error:", e)
    scan = (pd.DataFrame(rows).merge(universe[["Symbol", "Company", "Industry"]], on="Symbol", how="left")
            .set_index("Symbol"))
    scan["_o"] = scan["Action"].map(ORDER)
    scan = scan.sort_values(["_o", "Sharpe"], ascending=[True, False]).drop(columns="_o")
    scan["⭐"] = ""
    for s in scan.index[scan["Action"] == "🟢 BUY"]:
        for k, v in fundamentals(s).items():
            scan.loc[s, k] = v
        if scan.loc[s, "Strategy works?"] == "✅" and scan.loc[s, "Profitable?"] == "✅":
            scan.loc[s, "⭐"] = "⭐"
    return scan, latest


def save_results(scan, latest, mine=None, out="scanner_output"):
    os.makedirs(out, exist_ok=True)
    day = latest.date().isoformat()
    scan.to_csv(f"{out}/scan_latest.csv")
    if mine is not None and len(mine):
        mine.to_csv(f"{out}/my_holdings_plan.csv")
    log_path = f"{out}/signal_log.csv"
    cols = ["Action", "Company", "Close", "Buy limit", "Qty", "Stop-loss", "Entry", "P/L %", "Strategy works?"]
    new = scan[scan["Action"].isin(["🟢 BUY", "🔴 SELL", "🛑 STOPPED"])].reset_index()[["Symbol"] + cols]
    new.insert(0, "Signal date", day)
    if os.path.exists(log_path):
        old = pd.read_csv(log_path)
        new = pd.concat([old[old["Signal date"] != day], new], ignore_index=True)
    new.to_csv(log_path, index=False)
    return f"{out}/scan_latest.csv"


def _rs(x):
    return "-" if x is None or (isinstance(x, float) and np.isnan(x)) or pd.isna(x) else f"₹{x:,.2f}"


def ist_now():
    return pd.Timestamp.now(tz="Asia/Kolkata")


def build_message(scan, latest, mine=None):
    nxt = next_trading_day(latest).strftime("%a %d-%b")
    L = [f"📡 NSE PLAN for {nxt}  (data: {latest.date()} close, {UNIVERSE}: {len(scan)} stocks)",
         f"Capital ₹{CAPITAL:,.0f} | {MAX_POSITIONS} slots of ₹{SLOT:,.0f} | max loss/trade ₹{RISK_BUDGET:,.0f}"]
    if latest.date() < ist_now().date() and ist_now().hour >= 16:
        L.append("ℹ️ No new candle today (holiday/data delay). Plan is from the last trading day.")

    if mine is not None and len(mine):
        L.append(f"\n💼 YOUR HOLDINGS: {len(mine)}")
        for s, r in mine.iterrows():
            a = r["Action"]
            if a == "🟡 HOLD":
                L.append(f"🟡 {s} day {int(r['Day #'])} | {r['P/L %']:+.1f}% ({r['R']:+.1f}R)\n"
                         f"   GTT stop → {_rs(r['Stop-loss'])} | SELL if closes < {_rs(r['Exit trigger'])}")
            else:
                L.append(f"{a} {s}: {r['Plan']}")

    buys = scan[scan["Action"] == "🟢 BUY"]
    L.append(f"\n🟢 BUY on {nxt}: {len(buys)}")
    for s, r in buys.iterrows():
        if not r["Qty"]:
            L.append(f"• {s} {_rs(r['Close'])}: 1 share exceeds your slot. Skip."); continue
        flag = r.get("⭐", "") or ("" if r["Strategy works?"] == "✅" else "(weak history)")
        L.append(f"{flag} {s}  ({r.get('Industry', '') or ''})\n"
                 f"   ⏰ 9:15-9:30 AM, LIMIT, CNC → buy {int(r['Qty'])} @ max {_rs(r['Buy limit'])}\n"
                 f"   🛑 GTT stop ≈ {_rs(r['Stop-loss'])} | max loss ₹{r['Risk ₹']:,.0f}")

    for act, title in [("🔴 SELL", f"🔴 STRATEGY SELLS (open {nxt}, only if you own them)"),
                       ("🛑 STOPPED", "🛑 STOPS HIT today (only if you own them)")]:
        part = scan[scan["Action"] == act]
        if len(part):
            L.append(f"\n{title}: " + ", ".join(part.index[:30]))

    near = scan[scan["Action"] == "👀 NEAR BUY"].sort_values("Trigger dist %")
    L.append(f"\n👀 WATCHLIST: BUY fires if it CLOSES above the trigger ({len(near)})")
    for s, r in near.head(15).iterrows():
        L.append(f"• {s} close {_rs(r['Close'])} → trigger {_rs(r['BUY trigger'])} (+{r['Trigger dist %']:.1f}%)")
    if len(near) > 15:
        L.append(f"…{len(near) - 15} more in the CSV")

    holds = scan[scan["Action"] == "🟡 HOLD"]
    if mine is None or not len(mine):
        L.append(f"\n🟡 Strategy HOLDs: {len(holds)} (add your trades to my_holdings.csv to get personal exit plans)")
    L.append(f"\n⚪ WAIT: {scan['Action'].str.contains('WAIT|EXITED').sum()}")
    L.append("\n⭐ = strategy worked on this stock + company profitable"
             "\nLive checks run ~10:30 AM, 12:30 PM and 2:40 PM.\nEducational signals, not financial advice.")
    return "\n".join(L)


def split_message(text, limit=3900):
    parts, cur = [], ""
    for line in text.split("\n"):
        if len(cur) + len(line) + 1 > limit:
            parts.append(cur); cur = ""
        cur += line + "\n"
    if cur.strip():
        parts.append(cur)
    return parts


def send_telegram(text, files=()):
    import requests
    token, chat = os.getenv("TELEGRAM_TOKEN"), os.getenv("TELEGRAM_CHAT_ID")
    if not (token and chat):
        print("Telegram not configured, skipping"); return
    base = f"https://api.telegram.org/bot{token}"
    for part in split_message(text):
        r = requests.post(f"{base}/sendMessage", data={"chat_id": chat, "text": part}, timeout=30)
        print("Telegram message:", r.status_code)
    for path, name in files:
        with open(path, "rb") as f:
            requests.post(f"{base}/sendDocument", data={"chat_id": chat}, files={"document": (name, f)}, timeout=60)


def send_email(text, files=()):
    user, pwd, to = os.getenv("EMAIL_USER"), os.getenv("EMAIL_PASS"), os.getenv("EMAIL_TO")
    if not (user and pwd and to):
        print("Email not configured, skipping"); return
    import smtplib
    from email.message import EmailMessage
    m = EmailMessage()
    m["Subject"], m["From"], m["To"] = text.splitlines()[0], user, to
    m.set_content(text)
    for path, name in files:
        with open(path, "rb") as f:
            m.add_attachment(f.read(), maintype="text", subtype="csv", filename=name)
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as s:
        s.login(user, pwd); s.send_message(m)
    print("Email sent")


# ------------------------------ evening run ------------------------------
def evening():
    universe = load_universe(UNIVERSE)
    syms = list(dict.fromkeys(universe["Symbol"].tolist() + load_holdings()["Symbol"].tolist()))
    prices = download_all(syms)
    scan, latest = run_scan(universe, prices)
    mine = run_holdings(prices)
    csv_path = save_results(scan, latest, mine)
    msg = build_message(scan, latest, mine)
    print(msg)
    files = [(csv_path, "nse_plan.csv")] + ([("scanner_output/my_holdings_plan.csv", "my_holdings_plan.csv")] if len(mine) else [])
    send_telegram(msg, files)
    send_email(msg, files)


# ------------------------------ live (intraday) run ------------------------------
def live_prices(symbols):
    """Latest traded price per symbol from 5-minute bars (Yahoo data may lag by a few minutes)."""
    import yfinance as yf
    out = {}
    if not symbols:
        return out, None
    raw = yf.download([s + ".NS" for s in symbols], period="1d", interval="5m", group_by="ticker",
                      auto_adjust=True, progress=False, threads=True)
    stamp = None
    for s in symbols:
        try:
            c = raw[s + ".NS"]["Close"].dropna()
            if len(c):
                out[s] = float(c.iloc[-1]); stamp = c.index[-1] if stamp is None else max(stamp, c.index[-1])
        except Exception:
            pass
    return out, stamp


def live_alerts(levels, mine, prices):
    """Compare live prices with tonight's trigger levels. Returns list of (key, text)."""
    alerts = []
    for s, r in levels.iterrows():
        p = prices.get(s)
        if p is None or pd.isna(r.get("BUY trigger")):
            continue
        if p >= r["BUY trigger"] and p < r["RSI cap"]:
            alerts.append((f"{s}|buyform", f"⚡ BUY FORMING: {s} at ₹{p:,.2f} is above its trigger ₹{r['BUY trigger']:,.2f}.\n"
                           f"   Option A (the tested rule): if it CLOSES above ₹{r['BUY trigger']:,.2f}, tonight's plan says BUY → buy tomorrow 9:15 AM.\n"
                           f"   Option B (early, NOT backtested): buy after 3:15 PM today if still above it. Riskier: the signal can vanish by 3:30."))
    for s, r in (mine.iterrows() if mine is not None and len(mine) else []):
        p = prices.get(s)
        if p is None or r.get("Action") != "🟡 HOLD":
            continue
        stop, ex = r["Stop-loss"], r["Exit trigger"]
        if p <= stop:
            alerts.append((f"{s}|stop", f"🛑 {s} ₹{p:,.2f} touched your stop ₹{stop:,.2f}. Check that your GTT sold. If not, SELL now."))
        elif p <= stop * 1.015:
            alerts.append((f"{s}|nearstop", f"⚠️ {s} ₹{p:,.2f} is within 1.5% of your stop ₹{stop:,.2f}. Make sure the GTT is active. Don't cancel it."))
        if not pd.isna(ex) and p < ex:
            alerts.append((f"{s}|exit", f"🔴 {s} ₹{p:,.2f} is below its trend-exit price ₹{ex:,.2f}.\n"
                           f"   If it CLOSES below ₹{ex:,.2f}, the rule is SELL tomorrow at 9:15 AM. Nothing to do yet."))
    return alerts


def intraday(force=False, out="scanner_output"):
    now = ist_now()
    if not force and (now.weekday() >= 5 or not (dt.time(9, 20) <= now.time() <= dt.time(15, 30))):
        print("Market closed, skipping live check"); return
    lv_path, mine_path = f"{out}/scan_latest.csv", f"{out}/my_holdings_plan.csv"
    if not os.path.exists(lv_path):
        print("No evening plan yet. Run the evening scan first."); return
    levels = pd.read_csv(lv_path, index_col="Symbol")
    levels = levels[levels["Action"] == "👀 NEAR BUY"]
    mine = pd.read_csv(mine_path, index_col="Symbol") if os.path.exists(mine_path) else pd.DataFrame()
    syms = list(dict.fromkeys(levels.index.tolist() + mine.index.tolist()))
    prices, stamp = live_prices(syms)
    if not force and (stamp is None or pd.Timestamp(stamp).date() != now.date()):
        print("No trades today yet (holiday?), skipping"); return
    sent_path = f"{out}/live_sent.json"
    try:
        sent = json.load(open(sent_path))
    except Exception:
        sent = {}
    today = now.date().isoformat()
    sent = {today: sent.get(today, [])}
    new = [(k, t) for k, t in live_alerts(levels, mine, prices) if k not in sent[today]]
    tag = "PRE-CLOSE CHECK" if now.time() >= dt.time(14, 15) else "LIVE CHECK"
    print(f"{tag}: {len(prices)} prices, {len(new)} new alerts")
    if new:
        head = f"⏱ {tag} {now:%a %d-%b %H:%M} IST (prices may lag a few minutes)\n"
        send_telegram(head + "\n\n".join(t for _, t in new))
        sent[today] += [k for k, _ in new]
    json.dump(sent, open(sent_path, "w"))
    return new

## 4. Your holdings
Add every stock you actually bought. The planner then tells you exactly when to sell each one.

In [ ]:
# The stocks YOU actually bought. One line each: (symbol, buy date, buy price, quantity). Leave empty if none.
MY_HOLDINGS = [
    # ("BSE", "2026-11-05", 3512.00, 5),
]
pd.DataFrame(MY_HOLDINGS, columns=["Symbol", "BuyDate", "BuyPrice", "Qty"]).to_csv(HOLDINGS_FILE, index=False)
print(f"{len(MY_HOLDINGS)} holdings saved")

## 5. Scan all stocks

In [ ]:
universe = load_universe(UNIVERSE)
syms = list(dict.fromkeys(universe["Symbol"].tolist() + load_holdings()["Symbol"].tolist()))
prices = download_all(syms)                      # Nifty 500 takes ~3-5 minutes
scan, latest = run_scan(universe, prices)
mine = run_holdings(prices)
print(f"\nPlan for {next_trading_day(latest):%a %d-%b}  (based on {latest.date()} close)")
print(scan["Action"].value_counts().to_string())

## 6. Tomorrow's plan

In [ ]:
from IPython.display import display, Markdown


def show(df, title, cols=None):
    display(Markdown(f"### {title} ({len(df)})"))
    if not len(df):
        print("None."); return
    for s, r in df.iterrows():
        extra = (f"  \n<sub>strategy history on {s}: CAGR {r['Strategy CAGR %']:.1f}%/yr, win {r['Win %']:.0f}%, "
                 f"avg hold {r['Avg hold days']:.0f} days, works? {r['Strategy works?']}, turnover ₹{r['Turnover ₹cr']:.0f} cr/day</sub>"
                 if "Strategy CAGR %" in r else "")
        display(Markdown(f"**{r.get('⭐', '') or ''} {s}** {r['Action']}  \n{r['Plan']}{extra}"))


if len(mine):
    show(mine, "💼 YOUR HOLDINGS: exactly when to sell")
show(scan[scan["Action"] == "🟢 BUY"], "🟢 BUY tomorrow")
show(scan[scan["Action"] == "👀 NEAR BUY"].sort_values("Trigger dist %"), "👀 Watchlist: BUY fires if it CLOSES above the trigger")
for a in ("🔴 SELL", "🛑 STOPPED"):
    part = scan[scan["Action"] == a]
    print(f"\n{a} (strategy, only matters if you own them): {', '.join(part.index) or 'none'}")

## 7. ⏱ Live check (run any time 9:15 AM – 3:30 PM)
Compares live prices with tonight's trigger levels: BUYs forming on the watchlist, holdings near their stop-loss or below their trend-exit price.

In [ ]:
# Run this cell ANY TIME during market hours (9:15 AM - 3:30 PM) for a live check.
watch = scan[scan["Action"] == "👀 NEAR BUY"]
syms = list(dict.fromkeys(watch.index.tolist() + (mine.index.tolist() if len(mine) else [])))
live, stamp = live_prices(syms)
print(f"Live prices for {len(live)} stocks, last bar {stamp} (Yahoo data can lag a few minutes)\n")
rows = []
for s in watch.index:
    if s in live:
        r = watch.loc[s]
        rows.append({"Stock": s, "Live ₹": round(live[s], 2), "BUY trigger ₹": r["BUY trigger"],
                     "Gap to trigger %": round((r["BUY trigger"] / live[s] - 1) * 100, 2),
                     "Status": "⚡ above trigger" if live[s] >= r["BUY trigger"] else "below"})
if rows:
    display(pd.DataFrame(rows).sort_values("Gap to trigger %").set_index("Stock"))
alerts = live_alerts(watch, mine, live)
print("\n".join(t for _, t in alerts) or "No alerts right now.")

## 8. Deep dive into one stock

In [ ]:
import matplotlib.pyplot as plt


def deep_dive(sym, months=12):
    """Chart + past trades + exact trigger levels for one stock."""
    df = add_indicators(prices[sym])
    eq, tr, pos, skipped = backtest(df)
    plan = trade_plan(sym, df, tr, pos)
    d = df.iloc[-months * 21:]
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
    a1.plot(d.index, d["Close"], color="#333", lw=1.2, label="Close")
    a1.plot(d.index, d["ema_fast"], color="#1f77b4", lw=1, label=f"EMA{CONFIG['ema_fast']}")
    a1.plot(d.index, d["ema_slow"], color="#ff7f0e", lw=1, label=f"EMA{CONFIG['ema_slow']}")
    t = tr[tr["entry_date"] >= d.index[0]] if len(tr) else tr
    if len(t):
        a1.scatter(t["entry_date"], t["entry"], marker="^", s=90, c="green", zorder=5, label="Buy")
        a1.scatter(t["exit_date"], t["exit"], marker="v", s=90, c="red", zorder=5, label="Sell/Stop")
    trig = cross_price(df.iloc[-1])
    a1.axhline(trig, color="purple", ls=":", lw=1.2, label=f"{'Exit' if df['ema_fast'].iloc[-1] > df['ema_slow'].iloc[-1] else 'BUY'} trigger ₹{trig:.0f}")
    if pos:
        a1.axhline(pos["stop_tomorrow"], color="red", ls="--", lw=1, label=f"Stop ₹{pos['stop_tomorrow']:.0f}")
    a1.set_title(f"{sym}: {plan['Action'].split(' ', 1)[1]}"); a1.legend(loc="upper left"); a1.grid(alpha=.3)
    a2.plot(d.index, d["rsi"], color="purple"); a2.axhline(70, color="red", ls=":"); a2.axhline(30, color="green", ls=":")
    a2.set_ylabel("RSI"); a2.grid(alpha=.3)
    plt.tight_layout(); plt.show()
    print("PLAN:", plan["Plan"])
    print({k: round(v, 2) for k, v in stats(eq, tr, df).items()}, f"| entries skipped (gap-up): {skipped}")
    return tr.tail(10)


deep_dive(scan.index[0])        # change to any symbol, e.g. deep_dive("TCS")

## 9. Save results

In [ ]:
SAVE_TO_DRIVE = False           # True = keep a daily history in Google Drive
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/algo_scanner"
else:
    OUT = "scanner_output"
print("Saved:", save_results(scan, latest, mine, OUT), "+ signal_log.csv")

## 10. Full table

In [ ]:
scan